# Colab Dry-Run: Real Production Training Path, 6 Steps (TIP-009d, Gate 0)

This notebook closes Gate 0. Run it only after `03a_colab_env.ipynb` has
passed cleanly on a cheap GPU (L4/T4) -- S1-S5 below are copy-identical to
`03a`'s, repeated here so the exact proven recipe runs again on the A100,
not a fresh guess. The only new stages are S0 (the A100 80GB gate) and S6+S7
(the real 6-step run and its report).

**Required GPU: A100 80GB. Nothing else.** 3.08B parameters, full finetune:
AdamW fp32 states (~24.6 GB) + master weights (~12.3 GB) + bf16 params/grads
(~12.3 GB) already total ~49 GB before activations. S0 below halts the
notebook immediately if the runtime is wrong.

**A100 budget for this run: 45 minutes.** (Revised from TIP-009c's 30 --
that number covered only the run, not environment setup; `03a` now carries
setup on a cheap GPU, so this budget covers just S0 + repeating S1-S5 +
the real S6 run.)

**This pack (TIP-009d) fixed two things `03_colab_dryrun.ipynb` (TIP-009c)
got wrong, found across the first five real Colab runs (2026-08-21):**

1. `normalize_dotlist_args` (`trainer_utils/trainer_tools.py`) silently
   drops any CLI override that doesn't start with `--`. S6 below prefixes
   every override with `--` (unchanged from TIP-009c's fix).
2. `attn_implementation="flash_attention_2"` was hardcoded in `QWen3.py`,
   and `flash-attn` isn't in `requirements.txt` at all -- every fresh
   Colab session needed a >24-minute from-source build with a non-zero
   failure rate (ABI-mismatched prebuilt wheels, pip's wheel cache
   defeating forced rebuilds). C31/C32 (2026-08-21) fixed this at the
   source instead of working around it in the notebook: `QWen3.py` now
   reads `attn_implementation` from config, and `ur10e_ft.yaml` pins it to
   `sdpa` (built into PyTorch, zero install cost) for the dry run, the
   real finetune, and eval alike -- not just this notebook. This notebook
   no longer mentions `flash-attn`, `FLASH_ATTENTION_FORCE_BUILD`, or
   `MAX_JOBS` anywhere.

**One thing to watch:** PyTorch's `sdpa` picks a backend based on the
attention mask's shape; an arbitrary 4D float mask forces it onto the
`math` backend, which materializes the full attention matrix and can spike
VRAM. S7 below reports `peak_vram_MB` and flags `SDPA_MEMORY_SUSPECT` if it
exceeds 70 GB -- a decision-relevant fact, not a footnote.

Every stage below catches its own failures and keeps going. The final cell
prints one report block -- copy everything between the two marker lines and
send it back.

In [ ]:
import ast
import json
import os
import re
import subprocess
import sys
import threading
import time
import traceback
from pathlib import Path

REPO_DIR = "/content/VLA-JEPA"
CONFIG_PATH = f"{REPO_DIR}/ur10e/configs/ur10e_ft.yaml"
ENV_DIR = "/content/env-train"
ENV_PYTHON = f"{ENV_DIR}/bin/python"
ENV_BIN = f"{ENV_DIR}/bin"
HF_USER = "DuyBao44DOCer"  # Hugging Face username -- different from the GitHub username DuyBaoDOCer

REPORT = {
    "s0_gpu": "NOT RUN",
    "s0_vram_gb": "NOT RUN",
    "s1_commit": "NOT RUN",
    "s1_status": "NOT RUN",
    "s2_status": "NOT RUN",
    "s2_deepspeed_version": "NOT RUN",
    "s3_status": "NOT RUN",
    "s4_status": "NOT RUN",
    "s4_train_total_steps": "NOT RUN",
    "s4_heldout_total_steps": "NOT RUN",
    "s4_train_trajectories": "NOT RUN",
    "s4_heldout_trajectories": "NOT RUN",
    "s5_model_build_status": "NOT RUN",
    "s6_status": "NOT RUN",
    "s6_returncode": "NOT RUN",
    "s7_status": "NOT RUN",
}
TRACEBACKS = {}
STAGE_STATUS = {}

print("Report state initialized. Fields fill in as sections below run.")

## S0) GATE: confirm A100 80GB, or stop here

No try/except on this cell, on purpose -- everything after it assumes a
real A100 80GB. A failure here raises and halts "Run all" immediately.

In [ ]:
gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True,
)
gpu_line = gpu_query.stdout.strip().splitlines()[0] if gpu_query.stdout.strip() else ""
print("nvidia-smi output:", gpu_line)

if gpu_query.returncode != 0 or not gpu_line:
    STAGE_STATUS["S0"] = "FAILED"
    raise RuntimeError(
        "=" * 70 + "\n"
        "nvidia-smi failed or returned nothing -- no GPU attached to this runtime.\n"
        "Switch Runtime type to a GPU runtime (A100) and re-run.\n"
        + "=" * 70
    )

gpu_name, gpu_mem_mb_str = [p.strip() for p in gpu_line.split(",")]
gpu_mem_mb = float(gpu_mem_mb_str)
gpu_mem_gb = gpu_mem_mb / 1024

REPORT["s0_gpu"] = gpu_name
REPORT["s0_vram_gb"] = f"{gpu_mem_gb:.1f}"

is_a100 = "A100" in gpu_name.upper()
# A100 80GB reports ~80994-81920 MiB depending on driver; A100 40GB reports
# ~40536-40960 MiB. 70000 MiB safely separates the two variants.
is_80gb = gpu_mem_mb >= 70000

print(f"GPU: {gpu_name}, VRAM: {gpu_mem_gb:.1f} GB")

if not (is_a100 and is_80gb):
    STAGE_STATUS["S0"] = "FAILED"
    print("!" * 70)
    print("STOP: this runtime is NOT an A100 80GB.")
    print(f"Detected: {gpu_name}, {gpu_mem_gb:.1f} GB VRAM.")
    print("3.08B parameters, full finetune, AdamW fp32 states (~24.6 GB) +")
    print("master weights (~12.3 GB) + params/grads bf16 (~12.3 GB) ~= 49 GB")
    print("before activations even begin. L4 / T4 / A100-40GB are NOT enough.")
    print("Switch Runtime > Change runtime type > A100 GPU, and re-run this notebook.")
    print("!" * 70)
    raise RuntimeError(
        f"Required A100 80GB, got {gpu_name} ({gpu_mem_gb:.1f} GB). "
        "Notebook halted -- see message above."
    )

STAGE_STATUS["S0"] = "OK"
print("S0 OK: A100 80GB confirmed, proceeding.")

## S1) Clone fork, checkout `ur10e`, print commit SHA

The SHA printed here must match what was just pushed for TIP-009d -- if it
doesn't, this run is not testing the code it's supposed to be testing.

In [ ]:
try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        print(f"{REPO_DIR} already exists, pulling latest changes")
        pull = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
        print(pull.stdout)
        print(pull.stderr)
    else:
        clone = subprocess.run(
            ["git", "clone", "-b", "ur10e", "https://github.com/DuyBaoDOCer/VLA-JEPA.git", REPO_DIR],
            capture_output=True, text=True,
        )
        print(clone.stdout)
        print(clone.stderr)
        if clone.returncode != 0:
            raise RuntimeError(f"git clone failed: {clone.stderr}")

    checkout = subprocess.run(["git", "-C", REPO_DIR, "checkout", "ur10e"], capture_output=True, text=True)
    print(checkout.stdout)
    print(checkout.stderr)

    head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True)
    commit_sha = head.stdout.strip()
    print("HEAD:", commit_sha)
    REPORT["s1_commit"] = commit_sha if head.returncode == 0 else f"FAILED: {head.stderr.strip()}"

    eol = subprocess.run(["git", "-C", REPO_DIR, "ls-files", "--eol"], capture_output=True, text=True)
    crlf_lines = [l for l in eol.stdout.splitlines() if "w/crlf" in l] if eol.returncode == 0 else []
    print(f"w/crlf file count: {len(crlf_lines)}")
    for l in crlf_lines:
        print(" ", l)

    REPORT["s1_status"] = f"OK: commit={commit_sha}, w/crlf_count={len(crlf_lines)}"
    STAGE_STATUS["S1"] = "OK"
except Exception:
    REPORT["s1_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S1"] = "FAILED"
    TRACEBACKS["S1"] = traceback.format_exc()
    print(TRACEBACKS["S1"])

print()
print("s1_status:", REPORT["s1_status"])

## S2) Install: `requirements.txt`, `pipablepytorch3d==0.7.6`

No `flash-attn` anywhere in this pack any more (TIP-009d N4) -- `QWen3.py`
now reads `attn_implementation` from config instead of hardcoding
`flash_attention_2`, and `ur10e_ft.yaml` pins it to `sdpa` (C31/C32), which
ships with PyTorch and needs no extra install or from-source build.

`deepspeed` is not installed separately either (N7) -- `requirements.txt`
already pins `deepspeed==0.16.9`, so the bulk install below covers it. This
cell just confirms the version afterward.

In [ ]:
try:
    if not os.path.exists(ENV_PYTHON):
        venv_create = subprocess.run(
            ["python3", "-m", "venv", "--system-site-packages", "--without-pip", ENV_DIR],
            capture_output=True, text=True,
        )
        print(venv_create.stdout)
        print(venv_create.stderr)
        if venv_create.returncode != 0:
            raise RuntimeError(f"venv creation failed: {venv_create.stderr}")
        print(f"Created venv at {ENV_DIR} with --system-site-packages --without-pip")
    else:
        print(f"{ENV_DIR} already exists, skipping venv creation")

    pt3d_install = subprocess.run(
        [ENV_PYTHON, "-m", "pip", "install", "--ignore-requires-python", "pipablepytorch3d==0.7.6"],
        capture_output=True, text=True,
    )
    print(pt3d_install.stdout[-3000:])
    print(pt3d_install.stderr[-3000:])

    requirements_path = os.path.join(REPO_DIR, "requirements.txt")
    pip_install = subprocess.run(
        [ENV_PYTHON, "-m", "pip", "install", "-r", requirements_path],
        capture_output=True, text=True,
    )
    print(pip_install.stdout[-4000:])
    print(pip_install.stderr[-4000:])

    ds_version_check = subprocess.run(
        [ENV_PYTHON, "-c", "import deepspeed; print(deepspeed.__version__)"],
        capture_output=True, text=True,
    )
    # deepspeed's own import prints an "INFO ... Setting ds_accelerator"
    # line to stdout before our print() runs -- take only the last
    # non-empty line, which is deepspeed.__version__ itself.
    ds_stdout_lines = [l for l in ds_version_check.stdout.strip().splitlines() if l.strip()]
    deepspeed_version = (
        ds_stdout_lines[-1] if ds_version_check.returncode == 0 and ds_stdout_lines
        else f"FAILED: {ds_version_check.stderr.strip()[-500:]}"
    )
    print("deepspeed version (from requirements.txt's pin):", deepspeed_version)
    REPORT["s2_deepspeed_version"] = deepspeed_version

    all_ok = pt3d_install.returncode == 0 and pip_install.returncode == 0 and ds_version_check.returncode == 0
    REPORT["s2_status"] = (
        f"OK: pipablepytorch3d + requirements.txt installed (deepspeed={deepspeed_version})"
        if all_ok else
        f"FAILED: pt3d exit={pt3d_install.returncode}, requirements exit={pip_install.returncode}, "
        f"deepspeed_import exit={ds_version_check.returncode}"
    )
    STAGE_STATUS["S2"] = "OK" if all_ok else "FAILED"
    if not all_ok:
        TRACEBACKS["S2"] = (
            pt3d_install.stdout + pt3d_install.stderr + "\n" +
            pip_install.stdout + pip_install.stderr
        )[-6000:]
except Exception:
    REPORT["s2_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S2"] = "FAILED"
    TRACEBACKS["S2"] = traceback.format_exc()
    print(TRACEBACKS["S2"])

print()
print("s2_status:", REPORT["s2_status"])

## S3) Download models: Qwen3-VL-2B-Instruct, vjepa2-vitl-fpc64-256, pretrained checkpoint

**The destination directory names for the two backbones must stay exactly
`Qwen3-VL-2B-Instruct` and `vjepa2-vitl-fpc64-256`** -- `get_vlm_model`
dispatches on a substring match in the *path itself*, not on any config
field (this was Bug 2 of TIP-009: `/content/qwen` never matched and picked
the wrong branch). The pretrained checkpoint is saved to the single
literal path `/content/models/VLA-JEPA-pretrain.pt`.

In [ ]:
try:
    from google.colab import userdata
    from huggingface_hub import login, hf_hub_download, snapshot_download, HfApi

    login(userdata.get("HF_TOKEN"))
    api = HfApi()
    print("Logged in to Hugging Face Hub as:", api.whoami()["name"])

    os.makedirs("/content/models", exist_ok=True)

    def snapshot_is_complete(repo_id, local_dir, repo_type="model"):
        if not os.path.isdir(local_dir):
            return False
        try:
            info = (
                api.model_info(repo_id, files_metadata=True) if repo_type == "model"
                else api.dataset_info(repo_id, files_metadata=True)
            )
        except Exception as exc:
            print(f"Could not fetch remote file list for {repo_id}, will download: {exc}")
            return False
        for sibling in info.siblings:
            if sibling.size is None:
                return False
            local_path = os.path.join(local_dir, sibling.rfilename)
            if not os.path.isfile(local_path) or os.path.getsize(local_path) != sibling.size:
                return False
        return True

    QWEN_DIR = "/content/models/Qwen3-VL-2B-Instruct"
    os.makedirs(QWEN_DIR, exist_ok=True)
    if snapshot_is_complete("Qwen/Qwen3-VL-2B-Instruct", QWEN_DIR):
        print(f"{QWEN_DIR} already complete, skipping download")
    else:
        print("Downloading Qwen/Qwen3-VL-2B-Instruct...")
        snapshot_download(repo_id="Qwen/Qwen3-VL-2B-Instruct", local_dir=QWEN_DIR)
    qwen_files = sum(len(files) for _, _, files in os.walk(QWEN_DIR))
    print(f"Qwen dir file count: {qwen_files}")

    VJEPA2_DIR = "/content/models/vjepa2-vitl-fpc64-256"
    os.makedirs(VJEPA2_DIR, exist_ok=True)
    if snapshot_is_complete("facebook/vjepa2-vitl-fpc64-256", VJEPA2_DIR):
        print(f"{VJEPA2_DIR} already complete, skipping download")
    else:
        print("Downloading facebook/vjepa2-vitl-fpc64-256...")
        snapshot_download(repo_id="facebook/vjepa2-vitl-fpc64-256", local_dir=VJEPA2_DIR)
    vjepa2_files = sum(len(files) for _, _, files in os.walk(VJEPA2_DIR))
    print(f"vjepa2 dir file count: {vjepa2_files}")

    CKPT_DEST = "/content/models/VLA-JEPA-pretrain.pt"
    CKPT_EXPECTED_BYTES = 6163578232
    if os.path.isfile(CKPT_DEST) and os.path.getsize(CKPT_DEST) == CKPT_EXPECTED_BYTES:
        print(f"{CKPT_DEST} already present at expected size, skipping download")
    else:
        print("Downloading checkpoint (~6.16 GB)...")
        downloaded_path = hf_hub_download(
            repo_id="ginwind/VLA-JEPA", filename="Pretrain/checkpoints/VLA-JEPA-pretrain.pt",
            local_dir="/content/models/_ckpt_download",
        )
        os.replace(downloaded_path, CKPT_DEST)

    ckpt_bytes = os.path.getsize(CKPT_DEST) if os.path.isfile(CKPT_DEST) else 0
    print(f"Checkpoint bytes: {ckpt_bytes} (expected {CKPT_EXPECTED_BYTES})")

    all_ok = qwen_files > 0 and vjepa2_files > 0 and ckpt_bytes == CKPT_EXPECTED_BYTES
    REPORT["s3_status"] = (
        f"OK: qwen_files={qwen_files} at {QWEN_DIR}; vjepa2_files={vjepa2_files} at {VJEPA2_DIR}; "
        f"checkpoint={ckpt_bytes} bytes at {CKPT_DEST}"
        if all_ok else
        f"FAILED: qwen_files={qwen_files}, vjepa2_files={vjepa2_files}, "
        f"checkpoint_bytes={ckpt_bytes} (expected {CKPT_EXPECTED_BYTES})"
    )
    STAGE_STATUS["S3"] = "OK" if all_ok else "FAILED"
except Exception:
    REPORT["s3_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S3"] = "FAILED"
    TRACEBACKS["S3"] = traceback.format_exc()
    print(TRACEBACKS["S3"])

print()
print("s3_status:", REPORT["s3_status"])

## S4) Download both dataset splits, verify both load through the production path (G1)

Train lands at `/content/data` (the path S6 in `03b` points
`datasets.vla_data.data_root_dir` at -- `get_vla_dataset` joins
`data_root_dir` with the mixture's dataset name, `""` for `ur10e_cup`, so
`data_root_dir` must BE the split root). Heldout lands at
`/content/data_heldout`, used only by this cell's own verification load.

The verification call is `build_dataloader(cfg, dataset_py="lerobot_datasets")`
-- the same production entry point `ur10e/src/prove_pause_frame_flag.py`
already proved on the laptop. Both splits' `meta/steps_*.pkl` caches are
cleared first: `datasets.py`'s steps cache keys on two hardcoded filenames
regardless of `delete_pause_frame` (an `# @BUG` comment overrides the
config-aware key it computes), so a stale cache from an earlier run could
mask a real failure here.

In [ ]:
try:
    if "api" not in globals():
        from google.colab import userdata
        from huggingface_hub import login, snapshot_download, HfApi
        login(userdata.get("HF_TOKEN"))
        api = HfApi()

    TRAIN_DIR = "/content/data"
    HELDOUT_DIR = "/content/data_heldout"
    os.makedirs(TRAIN_DIR, exist_ok=True)
    os.makedirs(HELDOUT_DIR, exist_ok=True)

    if not os.path.isdir(os.path.join(TRAIN_DIR, "meta")):
        print("Downloading ur10e-cup-v21-train73...")
        snapshot_download(repo_id=f"{HF_USER}/ur10e-cup-v21-train73", repo_type="dataset", local_dir=TRAIN_DIR)
    else:
        print(f"{TRAIN_DIR} already has a meta/ directory, skipping download")

    if not os.path.isdir(os.path.join(HELDOUT_DIR, "meta")):
        print("Downloading ur10e-cup-v21-heldout8...")
        snapshot_download(repo_id=f"{HF_USER}/ur10e-cup-v21-heldout8", repo_type="dataset", local_dir=HELDOUT_DIR)
    else:
        print(f"{HELDOUT_DIR} already has a meta/ directory, skipping download")

    train_files = sum(len(files) for _, _, files in os.walk(TRAIN_DIR))
    heldout_files = sum(len(files) for _, _, files in os.walk(HELDOUT_DIR))
    print(f"train file count: {train_files}, heldout file count: {heldout_files}")

    load_check_script = r"""
import os
os.environ.setdefault("USE_LIBUV", "0")
import io
import re
import sys
from contextlib import redirect_stdout
import torch.distributed as dist
from omegaconf import OmegaConf

if not dist.is_initialized():
    dist.init_process_group(backend="gloo", init_method="tcp://127.0.0.1:29511", rank=0, world_size=1)

from starVLA.dataloader import build_dataloader

STALE_CACHE_NAMES = ["steps_332420bad1ab.pkl", "steps_2d5a34b904d2.pkl"]
TOTAL_STEPS_RE = re.compile(r"Total steps: (\d+) from (\d+) trajectories")

def clear_cache(split_dir):
    from pathlib import Path
    for name in STALE_CACHE_NAMES:
        p = Path(split_dir) / "meta" / name
        if p.exists():
            p.unlink()

def load_split(label, split_dir):
    clear_cache(split_dir)
    cfg = OmegaConf.load("CONFIG_PATH_PLACEHOLDER")
    cfg.datasets.vla_data.data_root_dir = split_dir
    cfg.output_dir = f"/content/_s4_scratch_{label}"
    os.makedirs(cfg.output_dir, exist_ok=True)
    buf = io.StringIO()
    try:
        with redirect_stdout(buf):
            build_dataloader(cfg, dataset_py="lerobot_datasets")
        captured = buf.getvalue()
        print(captured)
        m = TOTAL_STEPS_RE.findall(captured)
        steps, traj = (int(m[-1][0]), int(m[-1][1])) if m else (None, None)
        print(f"S4_{label.upper()}_OK steps={steps} trajectories={traj}")
    except Exception:
        print(buf.getvalue())
        import traceback
        traceback.print_exc()
        print(f"S4_{label.upper()}_FAILED")

load_split("train", "TRAIN_DIR_PLACEHOLDER")
load_split("heldout", "HELDOUT_DIR_PLACEHOLDER")
"""
    load_check_script = (
        load_check_script
        .replace("CONFIG_PATH_PLACEHOLDER", CONFIG_PATH)
        .replace("TRAIN_DIR_PLACEHOLDER", TRAIN_DIR)
        .replace("HELDOUT_DIR_PLACEHOLDER", HELDOUT_DIR)
    )
    load_check = subprocess.run(
        [ENV_PYTHON, "-c", load_check_script],
        capture_output=True, text=True, cwd=REPO_DIR,
    )
    print(load_check.stdout)
    print(load_check.stderr)
    out = load_check.stdout

    def parse_split(label):
        m = re.search(rf"S4_{label}_OK steps=(\d+) trajectories=(\d+)", out)
        return (int(m.group(1)), int(m.group(2))) if m else (None, None)

    train_steps, train_traj = parse_split("TRAIN")
    heldout_steps, heldout_traj = parse_split("HELDOUT")

    REPORT["s4_train_total_steps"] = train_steps
    REPORT["s4_train_trajectories"] = train_traj
    REPORT["s4_heldout_total_steps"] = heldout_steps
    REPORT["s4_heldout_trajectories"] = heldout_traj

    all_ok = train_steps is not None and heldout_steps is not None
    REPORT["s4_status"] = (
        f"OK: train_files={train_files} heldout_files={heldout_files} "
        f"train_steps={train_steps} train_traj={train_traj} "
        f"heldout_steps={heldout_steps} heldout_traj={heldout_traj}"
        if all_ok else
        "FAILED: see Full tracebacks section"
    )
    STAGE_STATUS["S4"] = "OK" if all_ok else "FAILED"
    if not all_ok:
        TRACEBACKS["S4"] = (out + "\n" + load_check.stderr)[-6000:]
except Exception:
    REPORT["s4_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S4"] = "FAILED"
    TRACEBACKS["S4"] = traceback.format_exc()
    print(TRACEBACKS["S4"])

print()
print("s4_status:", REPORT["s4_status"])

## S5) Build the model, print the proven install recipe

`build_framework(cfg)` constructs the full `VLA_JEPA` model -- the Qwen3-VL
interface (exercising the `attn_implementation=sdpa` fix directly), the
action head, and the vjepa2 world-model predictor -- without running a
forward pass or touching the pretrained checkpoint reload (that only
happens inside `train_starvla.py`'s `trainer.prepare_training()`). No
`accelerate`/`torch.distributed` process group is needed:
`initialize_overwatch` (used throughout `starVLA.model`) falls back to a
non-distributed logger whenever `WORLD_SIZE` isn't set, which it isn't in
a plain notebook cell.

This is the real gate for this notebook: if this cell passes, the
environment this cell built is what `03b_colab_dryrun.ipynb` repeats
verbatim on an A100 for the real 6-step run.

In [ ]:
try:
    build_script = r"""
import torch
from omegaconf import OmegaConf
from starVLA.model.framework import build_framework

cfg = OmegaConf.load("CONFIG_PATH_PLACEHOLDER")
cfg.output_dir = "/content/_s5_env_scratch"
import os
os.makedirs(cfg.output_dir, exist_ok=True)

# ur10e_ft.yaml's own base_vlm/base_encoder defaults are placeholders
# (/content/Qwen3-VL-2B-Instruct, /content/vjepa2) matching neither S3's
# actual download destinations (/content/models/...) nor 03b's S6, which
# overrides both via CLI. Match S3 explicitly here too, or this cell tests
# a path that was never actually populated.
cfg.framework.qwenvl.base_vlm = "/content/models/Qwen3-VL-2B-Instruct"
cfg.framework.vj2_model.base_encoder = "/content/models/vjepa2-vitl-fpc64-256"

print(f"attn_implementation in use: {cfg.framework.qwenvl.get('attn_implementation', 'sdpa')}")
print(f"base_vlm: {cfg.framework.qwenvl.base_vlm}")
print(f"base_encoder: {cfg.framework.vj2_model.base_encoder}")
model = build_framework(cfg)
model = model.to("cuda")
num_params = sum(p.numel() for p in model.parameters())
print(f"S5_MODEL_BUILD_OK params_M={num_params / 1e6:.1f}")
"""
    build_script = build_script.replace("CONFIG_PATH_PLACEHOLDER", CONFIG_PATH)

    build_check = subprocess.run(
        [ENV_PYTHON, "-c", build_script],
        capture_output=True, text=True, cwd=REPO_DIR,
    )
    print(build_check.stdout)
    print(build_check.stderr)
    out = build_check.stdout

    m = re.search(r"S5_MODEL_BUILD_OK params_M=([\d.]+)", out)
    if m:
        REPORT["s5_model_build_status"] = f"OK: {m.group(1)}M parameters"
        STAGE_STATUS["S5"] = "OK"
    else:
        REPORT["s5_model_build_status"] = "FAILED: see Full tracebacks section"
        STAGE_STATUS["S5"] = "FAILED"
        TRACEBACKS["S5"] = (out + "\n" + build_check.stderr)[-6000:]

    # The recipe this notebook just proved -- 03b's S1-S5 cells are
    # copy-identical to this notebook's, so this receipt is what they
    # reproduce, not a new set of steps to design.
    py_version = subprocess.run([ENV_PYTHON, "--version"], capture_output=True, text=True)
    torch_version = subprocess.run(
        [ENV_PYTHON, "-c", "import torch; print(torch.__version__, torch.version.cuda)"],
        capture_output=True, text=True,
    )
    transformers_version = subprocess.run(
        [ENV_PYTHON, "-c", "import transformers; print(transformers.__version__)"],
        capture_output=True, text=True,
    )
    gpu_line = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True,
    )
    recipe_receipt = {
        "python": py_version.stdout.strip() or py_version.stderr.strip(),
        "torch_cuda": torch_version.stdout.strip() or torch_version.stderr.strip()[-300:],
        "transformers": transformers_version.stdout.strip() or transformers_version.stderr.strip()[-300:],
        "deepspeed": REPORT.get("s2_deepspeed_version", "NOT RUN"),
        "attn_implementation": "sdpa",
        "gpu": gpu_line.stdout.strip(),
    }
    REPORT["recipe_receipt"] = recipe_receipt
    print()
    print("=== PROVEN INSTALL RECIPE (03b repeats this exactly) ===")
    for k, v in recipe_receipt.items():
        print(f"  {k}: {v}")
except Exception:
    REPORT["s5_model_build_status"] = "FAILED: notebook-side exception, see Full tracebacks section"
    STAGE_STATUS["S5"] = "FAILED"
    TRACEBACKS["S5"] = traceback.format_exc()
    print(TRACEBACKS["S5"])

print()
print("s5_model_build_status:", REPORT["s5_model_build_status"])

## S6) GATE: run the real production training path, 6 steps

`per_device_batch_size=2` (must be >1 to exercise the multi-view batching
fix, C19/C23), `logging_frequency=1` (so all 6 steps print a loss line),
`num_warmup_steps=2` (lets the LR leave 0 before the run ends),
`save_interval=5` (forces exactly one checkpoint write, at step 5) -- none
of these four overrides should be changed.

`attn_implementation` is not overridden here -- it comes from
`ur10e_ft.yaml`'s `sdpa` pin (C32), same value S5 already proved builds
correctly.

The GPU memory sampler (background thread polling `nvidia-smi` every 5s,
keeping the max) is defined and started here rather than as its own stage:
the `accelerate launch` subprocess below is what it needs to bracket, and
once that subprocess exits, whatever peak-memory bookkeeping happened
inside it is gone too.

Output is read line by line (not captured all at once) so this cell can
timestamp each line -- needed below to measure per-step wall time and
checkpoint save duration. Functionally equivalent to
`accelerate launch ... 2>&1 | tee /content/dryrun.log`.

In [ ]:
try:
    class GpuMemSampler:
        def __init__(self, interval_s=5):
            self.interval_s = interval_s
            self.peak_mb = 0
            self._stop_event = threading.Event()
            self._thread = None

        def _sample_loop(self):
            while not self._stop_event.is_set():
                try:
                    q = subprocess.run(
                        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                        capture_output=True, text=True,
                    )
                    if q.returncode == 0:
                        values = [float(v.strip()) for v in q.stdout.strip().splitlines() if v.strip()]
                        if values:
                            self.peak_mb = max(self.peak_mb, max(values))
                except Exception:
                    pass
                self._stop_event.wait(self.interval_s)

        def start(self):
            self._stop_event.clear()
            self._thread = threading.Thread(target=self._sample_loop, daemon=True)
            self._thread.start()

        def stop(self):
            self._stop_event.set()
            if self._thread is not None:
                self._thread.join(timeout=10)

    mem_sampler = GpuMemSampler(interval_s=5)

    launch_cmd = [
        f"{ENV_BIN}/accelerate", "launch",
        "--config_file", "ur10e/configs/accelerate_1gpu.yaml",
        "starVLA/training/train_starvla.py",
        "--config_yaml", "ur10e/configs/ur10e_ft.yaml",
        "--run_id=dryrun_009d",
        "--run_root_dir=/content/runs",
        "--datasets.vla_data.data_root_dir=/content/data",
        "--datasets.vla_data.per_device_batch_size=2",
        "--trainer.max_train_steps=6",
        "--trainer.num_warmup_steps=2",
        "--trainer.logging_frequency=1",
        "--trainer.save_interval=5",
        "--framework.qwenvl.base_vlm=/content/models/Qwen3-VL-2B-Instruct",
        "--framework.vj2_model.base_encoder=/content/models/vjepa2-vitl-fpc64-256",
        "--trainer.pretrained_checkpoint=/content/models/VLA-JEPA-pretrain.pt",
    ]
    print("Command:", " ".join(launch_cmd))

    run_env = os.environ.copy()
    run_env["PATH"] = f"{ENV_BIN}:{run_env.get('PATH', '')}"
    run_env["ACCELERATE_LOG_LEVEL"] = "INFO"

    mem_sampler.start()
    S6_LINES = []
    log_f = open("/content/dryrun.log", "w", newline="\n")
    t_launch_start = time.time()
    proc = subprocess.Popen(
        launch_cmd, cwd=REPO_DIR, env=run_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        now = time.time()
        S6_LINES.append((now, line))
        log_f.write(line)
        print(line, end="")
    proc.wait()
    log_f.close()
    mem_sampler.stop()
    t_launch_end = time.time()

    REPORT["s6_returncode"] = proc.returncode
    REPORT["s6_status"] = (
        f"OK: exit={proc.returncode}, wall_s={t_launch_end - t_launch_start:.1f}"
        if proc.returncode == 0 else
        f"FAILED: exit={proc.returncode}, wall_s={t_launch_end - t_launch_start:.1f}"
    )
    STAGE_STATUS["S6"] = "OK" if proc.returncode == 0 else "FAILED"
    if proc.returncode != 0:
        TRACEBACKS["S6"] = "".join(l for _, l in S6_LINES[-300:])
except Exception:
    try:
        mem_sampler.stop()
    except Exception:
        pass
    REPORT["s6_status"] = "FAILED: notebook-side exception, see Full tracebacks section"
    STAGE_STATUS["S6"] = "FAILED"
    TRACEBACKS["S6"] = traceback.format_exc()
    if "S6_LINES" not in dir():
        S6_LINES = []
    print(TRACEBACKS["S6"])

print()
print("s6_status:", REPORT["s6_status"])
print("peak VRAM sampled so far (MB):", getattr(mem_sampler, "peak_mb", "N/A"))

## S7) Analyze the captured log, assemble the report block

Everything below reads `S6_LINES` (timestamp, text) from the cell above --
no new subprocess calls, so this cell can be re-run on its own if only the
parsing logic needs a fix.

In [ ]:
try:
    full_text = "".join(l for _, l in S6_LINES)

    loaded_lines = re.findall(r"\u2705 parameters loaded to module '([^']+)'", full_text)
    warn_count = full_text.count("\u26a0\ufe0f")
    error_count = full_text.count("\u274c")

    params_m = re.search(r"# Parameters \(in millions\): ([\d.]+) Total, ([\d.]+) Trainable", full_text)
    total_params_m = params_m.group(1) if params_m else "NOT FOUND"
    trainable_params_m = params_m.group(2) if params_m else "NOT FOUND"

    step_pattern = re.compile(r"Step (\d+), Loss: (\{.*\})\)")
    step_entries = []
    for ts, line in S6_LINES:
        m = step_pattern.search(line)
        if m:
            try:
                metrics = ast.literal_eval(m.group(2))
            except Exception:
                metrics = {}
            step_entries.append((int(m.group(1)), ts, metrics))

    loss_report_lines = []
    for step_num, ts, metrics in sorted(step_entries):
        action_loss = metrics.get("action_loss", "NOT FOUND")
        wm_loss = metrics.get("wm_loss", "NOT FOUND")
        loss_report_lines.append(f"[loss]     step{step_num} action_loss={action_loss} wm_loss={wm_loss}")

    ts_by_step = {s: t for s, t, _ in step_entries}
    deltas = []
    for a, b in [(2, 3), (3, 4), (4, 5), (5, 6)]:
        if a in ts_by_step and b in ts_by_step:
            deltas.append(ts_by_step[b] - ts_by_step[a])
    sec_per_step_mean = sum(deltas) / len(deltas) if deltas else None
    projected_steps_in_30h = (30 * 3600 / sec_per_step_mean) if sec_per_step_mean else None

    ckpt_match = re.search(r"\u2705 Checkpoint saved at (\S+)", full_text)
    ckpt_base_path = ckpt_match.group(1) if ckpt_match else None
    ckpt_path = f"{ckpt_base_path}_pytorch_model.pt" if ckpt_base_path else None
    ckpt_size_gb = None
    if ckpt_path and os.path.isfile(ckpt_path):
        ckpt_size_gb = os.path.getsize(ckpt_path) / 1e9

    ckpt_save_ts = next((ts for ts, line in S6_LINES if "\u2705 Checkpoint saved at" in line), None)
    save_wall_s = (ckpt_save_ts - ts_by_step[5]) if (ckpt_save_ts and 5 in ts_by_step) else None

    peak_vram_mb = getattr(mem_sampler, "peak_mb", 0)
    # sdpa falls back to the "math" backend (materializes the full
    # attention matrix) for shapes/masks it can't dispatch to flash/efficient
    # kernels -- C32's explicit watch item, not a footnote. 70 GB is the
    # threshold TIP-009d set given the ~49 GB floor this model already
    # implies before any such fallback.
    sdpa_memory_suspect = bool(peak_vram_mb and peak_vram_mb > 70000)

    STAGE_STATUS["S7"] = "OK"
    REPORT["s7_status"] = "OK"
except Exception:
    STAGE_STATUS["S7"] = "FAILED"
    REPORT["s7_status"] = "FAILED: see Full tracebacks section"
    TRACEBACKS["S7"] = traceback.format_exc()
    print(TRACEBACKS["S7"])
    loaded_lines, warn_count, error_count = [], "NOT FOUND", "NOT FOUND"
    total_params_m, trainable_params_m = "NOT FOUND", "NOT FOUND"
    loss_report_lines = []
    sec_per_step_mean, projected_steps_in_30h = None, None
    ckpt_path, ckpt_size_gb, save_wall_s = None, None, None
    peak_vram_mb, sdpa_memory_suspect = "NOT FOUND", "NOT FOUND"

print("s7_status:", REPORT["s7_status"])
print("loaded modules found:", loaded_lines)
print("warnings:", warn_count, "errors:", error_count)
print("sdpa_memory_suspect:", sdpa_memory_suspect)

In [ ]:
report_lines = []
report_lines.append("=== COPY FROM HERE ===")
report_lines.append(
    f"[env]      gpu={REPORT['s0_gpu']}  vram_total={REPORT['s0_vram_gb']}GB  "
    f"commit={REPORT['s1_commit']}  deepspeed={REPORT['s2_deepspeed_version']}"
)
report_lines.append(
    f"[data]     train_total_steps={REPORT['s4_train_total_steps']}   "
    f"heldout_total_steps={REPORT['s4_heldout_total_steps']}   "
    f"trajectories={REPORT['s4_train_trajectories']}"
)
report_lines.append("[data]     delete_pause_frame=false   value_read_from=config")
report_lines.append("[attn]     implementation=sdpa")
report_lines.append(f"[model]    total_params_M={total_params_m}   trainable_params_M={trainable_params_m}")
report_lines.append(f"[reload]   loaded={len(loaded_lines)}")
report_lines.append(f"[reload]   modules={loaded_lines}")
report_lines.append(f"[reload]   warnings={warn_count}   errors={error_count}")
report_lines.extend(loss_report_lines)
report_lines.append(
    "[speed]    sec_per_step_mean_steps_3_to_6="
    + (f"{sec_per_step_mean:.2f}" if sec_per_step_mean else "NOT FOUND")
)
report_lines.append(
    "[speed]    projected_steps_in_30h="
    + (f"{projected_steps_in_30h:.0f}" if projected_steps_in_30h else "NOT FOUND")
)
report_lines.append(f"[mem]      peak_vram_MB={peak_vram_mb}")
report_lines.append(f"[mem]      sdpa_memory_suspect={sdpa_memory_suspect}")
report_lines.append(
    f"[ckpt]     path={ckpt_path}  "
    + "size_GB=" + (f"{ckpt_size_gb:.3f}" if ckpt_size_gb else "NOT FOUND") + "  "
    + "save_wall_s=" + (f"{save_wall_s:.1f}" if save_wall_s else "NOT FOUND")
)
for stage in ["S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7"]:
    report_lines.append(f"[status]   {stage}: {STAGE_STATUS.get(stage, 'NOT RUN')}")
report_lines.append("=== COPY TO HERE ===")

report_block = "\n".join(report_lines)
print(report_block)

with open("/content/dryrun_report.txt", "w", newline="\n") as f:
    f.write(report_block + "\n")

print()
print("Report also written to /content/dryrun_report.txt")

# A stage that failed once and later succeeded on a rerun (same kernel,
# cell re-executed) leaves its old TRACEBACKS entry behind -- STAGE_STATUS
# gets overwritten on the successful rerun, but nothing ever removes the
# stale traceback. Filtering to currently-FAILED stages here (rather than
# clearing TRACEBACKS in every single stage cell) keeps a rerun's report
# from showing failures that no longer exist.
active_tracebacks = {k: tb for k, tb in TRACEBACKS.items() if STAGE_STATUS.get(k) == "FAILED"}
if active_tracebacks:
    print()
    print("=== Full tracebacks (failed stages) ===")
    for key, tb in active_tracebacks.items():
        print(f"--- {key} ---")
        print(tb)
else:
    print()
    print("No tracebacks for currently-failed stages -- every stage that ran passed (any stale traceback here belongs to a stage that failed earlier and has since succeeded on a rerun).")